# Build Train-Only Sentiment Features on Colab GPU

Use this notebook for the expensive `Movies_and_TV` / `Video_Games` review-text sentiment pass. It runs the same repository CLI as local development:

```bash
python -m src.features.build_sentiment_features --dataset movies_and_tv
```

The job writes local/reproducible artifacts under `data/processed/<dataset>/advanced_features/`:

- `train_sentiment.parquet`
- `train_sentiment_meta.json`
- `user_review_aggregates.parquet`
- `item_review_aggregates.parquet`

It does **not** run model evaluation. Run evaluation later after these artifacts are available.

## 1. Select GPU Runtime

In Colab: `Runtime` -> `Change runtime type` -> `T4 GPU`.

The repo code prefers `cuda -> mps -> cpu`, so Colab should report `cuda` in `train_sentiment_meta.json`.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2. Mount Drive and Point at Data

Put your data in Drive with this shape:

```text
MyDrive/amazon-hybrid-recsys-data/
  raw/
    Movies_and_TV.jsonl or Movies_and_TV.jsonl.gz
    meta_Movies_and_TV.jsonl or meta_Movies_and_TV.jsonl.gz
  processed/
    movies_and_tv/
      train.parquet
      metadata.parquet
```

The notebook symlinks the cloned repo's `data/raw` and `data/processed` to this Drive folder, so outputs persist after the Colab VM shuts down.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/amazon-hybrid-recsys-data'
DATASET = 'movies_and_tv'  # change to 'video_games' if needed
BRANCH = 'feat/advanced-models'

## 3. Clone Repo and Install Minimal Runtime Dependencies

This installs only the packages needed for sentiment feature building, not the full recommender stack.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run(*args):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run([str(arg) for arg in args], check=True, env=env)


REPO = Path('/content/amazon-hybrid-recsys')
if not REPO.exists():
    run('git', 'clone', 'https://github.com/Fgram-devAI/amazon-hybrid-recsys.git', REPO)
os.chdir(REPO)
print('cwd:', Path.cwd())
run('git', 'fetch', 'origin', BRANCH)
run('git', 'checkout', BRANCH)
run(sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'pyarrow', 'pyyaml', 'tqdm', 'requests', 'transformers', 'accelerate')

## 4. Link Drive Data Into the Repo

This does not copy the large files. It creates symlinks from repo-local `data/raw` and `data/processed` to your Drive-backed folders.

In [ ]:
import os
import shutil
from pathlib import Path

repo_data = REPO / 'data'
drive_data = Path(DRIVE_DATA)
repo_data.mkdir(exist_ok=True)

for name in ['raw', 'processed']:
    src = drive_data / name
    dst = repo_data / name
    if not src.exists():
        raise FileNotFoundError(f'Missing Drive data folder: {src}')
    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(dst, '->', src)

processed = repo_data / 'processed' / DATASET
required = [processed / 'train.parquet', processed / 'metadata.parquet']
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print('processed dataset ready:', processed)

## 5. Optional Smoke Run

Run this first if you want to verify paths, CUDA, and output schema before committing to the full job.

In [ ]:
run(sys.executable, '-u', '-m', 'src.features.build_sentiment_features', '--dataset', DATASET, '--max-rows', '5000')

## 6. Full Sentiment Feature Build

Run this after the smoke run works. It overwrites the smoke artifacts at the end of the job.

In [ ]:
run(sys.executable, '-u', '-m', 'src.features.build_sentiment_features', '--dataset', DATASET)

## 7. Validate Outputs

Check that the run used CUDA and produced the expected row count.

In [ ]:
import json
import pandas as pd
from pathlib import Path

af = Path('data/processed') / DATASET / 'advanced_features'
meta = json.loads((af / 'train_sentiment_meta.json').read_text())
sent = pd.read_parquet(af / 'train_sentiment.parquet')
users = pd.read_parquet(af / 'user_review_aggregates.parquet')
items = pd.read_parquet(af / 'item_review_aggregates.parquet')

print(json.dumps(meta, indent=2))
print('sentiment shape:', sent.shape)
print('user aggregates:', users.shape)
print('item aggregates:', items.shape)
print(sent['sentiment_label'].value_counts(dropna=False))
print(sent.head(10).to_string(index=False))

## 8. Bring Artifacts Back Local

Because `data/processed` points to Drive, the artifacts are already persisted there. Copy or sync this folder back to your local repo before running evaluation locally:

```text
data/processed/<dataset>/advanced_features/
```

Do not commit those artifacts.